# Qwen3 Proxy Label Fine-Tuning in Colab

This notebook fine-tunes an LLM on your candidate dataset using these labels:

- `0-30 -> low_match`
- `31-70 -> medium_match`
- `71-100 -> high_match`

Use this notebook with the Excel file `Potential_Talents_Proxy_Labels.xlsx`.

Recommended runtime:
- GPU runtime in Colab
- `Qwen/Qwen3Guard-Gen-0.6B`
- as per the memory I changed the model to a smaller one

## 1 - Install Required Libraries

This cell prepares the Colab environment by listing the Python packages needed for fine-tuning: Transformers for the model, TRL for supervised fine-tuning, PEFT for LoRA adapters, Accelerate and BitsAndBytes for efficient GPU training, Datasets for reading JSONL files, and OpenPyXL for reading Excel files. The install command is commented so you can enable it only when the Colab runtime does not already have the required packages.


In [ ]:
# Install the main libraries needed for fine-tuning in Colab.
# The command is commented out so you can run it only when your environment needs it.
#!pip -q install "transformers>=4.57.0" "trl>=0.24.0" "peft>=0.17.0" "accelerate>=1.10.0" "datasets>=4.0.0" "bitsandbytes>=0.47.0" "openpyxl>=3.1.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.8/760.8 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 22.0 MB/s eta 0:00:00


## 2 - Upload the Excel File and Set Configuration

This cell lets you upload `Potential_Talents_Proxy_Labels.xlsx` into Colab, then stores the uploaded file name in `INPUT_XLSX`. It also defines the base model, output folders, random seed, training hyperparameters, batch sizes, gradient accumulation, and maximum sequence length used later in the notebook.


In [ ]:
from google.colab import files

# Open the Colab file upload dialog and store the uploaded file name.
uploaded = files.upload()
INPUT_XLSX = list(uploaded.keys())[0]
print('Uploaded file:', INPUT_XLSX)

# Main model, output, and dataset paths used by the rest of the notebook.
MODEL_NAME = 'Qwen/Qwen3Guard-Gen-0.6B'
OUTPUT_DIR = '/content/qwen3_proxy_labels'
DATA_DIR = '/content/proxy_label_data'

# Training configuration. You can tune these values if Colab memory or runtime becomes a problem.
SEED = 42
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUMULATION = 16
MAX_LENGTH = 256

print('Model:', MODEL_NAME)


Saving Potential_Talents_Proxy_Labels.xlsx to Potential_Talents_Proxy_Labels (2).xlsx
Uploaded file: Potential_Talents_Proxy_Labels (2).xlsx
Model: Qwen/Qwen3Guard-Gen-0.6B


## 3 - Build the Training, Validation, and Test Datasets

This cell reads the Excel sheet, validates that the required columns exist, cleans text values, converts every candidate row into a chat-style training example, and creates stratified train/validation/test splits. It writes those splits as JSONL files so the Hugging Face `datasets` library and TRL trainer can read them easily.


In [ ]:
import json
import os
import random
from collections import Counter, defaultdict

from openpyxl import load_workbook

# The system message defines the role and forces the model to answer with one of the allowed labels.
SYSTEM_MESSAGE = (
    'You are a recruiting screening model. '
    'Read the candidate information and return only one label: '
    'low_match, medium_match, or high_match.'
)


def normalize_text(value):
    # Convert empty cells to empty strings and remove extra spaces from text values.
    if value is None:
        return ''
    return ' '.join(str(value).strip().split())


def load_rows(input_xlsx, sheet_name='Sheet1'):
    # Read the Excel sheet and map each required column name to its index.
    workbook = load_workbook(input_xlsx, data_only=True)
    sheet = workbook[sheet_name]
    rows = list(sheet.iter_rows(values_only=True))
    headers = [normalize_text(cell) for cell in rows[0]]
    header_index = {name: idx for idx, name in enumerate(headers)}

    # Stop early if the file does not contain the columns needed for training.
    required = {'id', 'title', 'location', 'screening_score', 'match_label'}
    missing = sorted(required - set(header_index))
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    records = []
    for raw_row in rows[1:]:
        title = normalize_text(raw_row[header_index['title']])
        location = normalize_text(raw_row[header_index['location']])
        match_label = normalize_text(raw_row[header_index['match_label']])

        # Skip rows that cannot teach the model because the title or target label is missing.
        if not title or not match_label:
            continue

        records.append({
            'id': raw_row[header_index['id']],
            'title': title,
            'location': location,
            'screening_score': raw_row[header_index['screening_score']],
            'match_label': match_label,
        })
    return records


def build_example(row):
    # Format each row as a chat-style supervised fine-tuning example.
    user_message = (
        'Classify the candidate into exactly one label: '
        'low_match, medium_match, or high_match.

'
        f"Candidate title: {row['title']}
"
        f"Candidate location: {row['location']}"
    )
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_MESSAGE},
            {'role': 'user', 'content': user_message},
        ],
        'completion': [
            {'role': 'assistant', 'content': row['match_label']},
        ],
        'metadata': {
            'id': row['id'],
            'screening_score': row['screening_score'],
        },
    }


def stratified_split(rows, train_ratio=0.8, validation_ratio=0.1, test_ratio=0.1, seed=42):
    # Group rows by label so each split keeps a similar label distribution.
    grouped = defaultdict(list)
    for row in rows:
        grouped[row['match_label']].append(row)

    rng = random.Random(seed)
    split_rows = {'train': [], 'validation': [], 'test': []}

    for label, items in grouped.items():
        rng.shuffle(items)
        n_items = len(items)
        n_train = int(n_items * train_ratio)
        n_validation = int(n_items * validation_ratio)
        n_test = n_items - n_train - n_validation

        # For labels with at least three examples, try to place at least one row in every split.
        if n_items >= 3:
            if n_train == 0:
                n_train = 1
            if n_validation == 0:
                n_validation = 1
                n_train = max(1, n_train - 1)
            if n_test == 0:
                n_test = 1
                if n_train > 1:
                    n_train -= 1
                else:
                    n_validation = max(1, n_validation - 1)

        split_rows['train'].extend(items[:n_train])
        split_rows['validation'].extend(items[n_train:n_train + n_validation])
        split_rows['test'].extend(items[n_train + n_validation:n_train + n_validation + n_test])

    return split_rows


def write_jsonl(path, rows):
    # Write one JSON training example per line, which is the format expected by load_dataset.
    with open(path, 'w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(build_example(row), ensure_ascii=False) + '
')


# Load the uploaded Excel file, split the rows, and save each split to disk.
rows = load_rows(INPUT_XLSX)
splits = stratified_split(rows, seed=SEED)

os.makedirs(DATA_DIR, exist_ok=True)
for split_name, split_rows in splits.items():
    write_jsonl(os.path.join(DATA_DIR, f'{split_name}.jsonl'), split_rows)

# Print a quick quality check so you can confirm the split sizes and label balance.
summary = {
    split_name: {
        'count': len(split_rows),
        'label_counts': dict(Counter(row['match_label'] for row in split_rows)),
    }
    for split_name, split_rows in splits.items()
}

print(json.dumps(summary, indent=2))
print('
Example training row:')
print(json.dumps(build_example(rows[0]), indent=2))


{
  "train": {
    "count": 1024,
    "label_counts": {
      "high_match": 600,
      "medium_match": 145,
      "low_match": 279
    }
  },
  "validation": {
    "count": 127,
    "label_counts": {
      "high_match": 75,
      "medium_match": 18,
      "low_match": 34
    }
  },
  "test": {
    "count": 130,
    "label_counts": {
      "high_match": 75,
      "medium_match": 19,
      "low_match": 36
    }
  }
}

Example training row:
{
  "prompt": [
    {
      "role": "system",
      "content": "You are a recruiting screening model. Read the candidate information and return only one label: low_match, medium_match, or high_match."
    },
    {
      "role": "user",
      "content": "Classify the candidate into exactly one label: low_match, medium_match, or high_match.\n\nCandidate title: innovative and driven professional seeking a role in data analyticsdata science in the information technology industry.\nCandidate location: United States"
    }
  ],
  "completion": [
    {
      

After this cell, the dataset files are saved in `DATA_DIR` as `train.jsonl`, `validation.jsonl`, and `test.jsonl`. The printed summary helps you confirm the number of rows in each split and whether the label distribution looks reasonable before training.


## 4 - Fine-Tune Qwen3Guard with QLoRA

This cell loads the prepared JSONL data, checks that a GPU is available, loads the tokenizer and base model in 4-bit precision, configures LoRA adapters, and starts supervised fine-tuning with TRL. QLoRA keeps the memory requirement lower by training small adapter weights instead of the full model.


In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

# QLoRA training needs a GPU runtime in Colab.
if not torch.cuda.is_available():
    raise RuntimeError('Please switch Colab to a GPU runtime first.')

# Use bfloat16 when the GPU supports it, otherwise fall back to float16.
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# Load the JSONL files created in the previous cell as Hugging Face datasets.
dataset = load_dataset(
    'json',
    data_files={
        'train': os.path.join(DATA_DIR, 'train.jsonl'),
        'validation': os.path.join(DATA_DIR, 'validation.jsonl'),
    },
)

# Load the tokenizer and make sure padding is configured for batching.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# Load the base model in 4-bit precision to reduce GPU memory usage.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map='auto',
    torch_dtype=compute_dtype,
    trust_remote_code=True,
)
model.config.use_cache = False

# LoRA trains a small set of adapter weights instead of updating the full model.
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj',
        'k_proj',
        'v_proj',
        'o_proj',
        'gate_proj',
        'up_proj',
        'down_proj',
    ],
)

# SFTConfig controls the training loop, evaluation, checkpointing, and precision settings.
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    max_length=MAX_LENGTH,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to='none',
    completion_only_loss=True,
)

# Create the trainer with the model, tokenizer, datasets, and LoRA configuration.
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    processing_class=tokenizer,
    peft_config=peft_config,
)

# Run fine-tuning, then save the adapter and tokenizer files for later inference.
trainer.train()
trainer.save_model(os.path.join(OUTPUT_DIR, 'final_adapter'))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, 'final_adapter'))

print('Training finished.')
print('Saved adapter to:', os.path.join(OUTPUT_DIR, 'final_adapter'))


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.60k [00:00<?, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/1024 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/127 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.000000,nan,0.000000,262144.000000,0.000000
2,0.000000,nan,0.000000,524288.000000,0.000000
3,0.000000,nan,0.000000,786432.000000,0.000000


Training finished.
Saved adapter to: /content/qwen3_proxy_labels/final_adapter




The fine-tuned LoRA adapter and tokenizer files are saved under `OUTPUT_DIR/final_adapter`. The training logs and validation evaluation at each epoch help you judge whether the model is learning from your proxy labels.


## 5 - Optional TorchAO Cleanup

This optional cell is only for troubleshooting package conflicts. If Colab shows an error related to `torchao`, you can uncomment and run these lines to remove that package from the environment.


In [ ]:
# Optional cleanup cell for environments where torchao causes package conflicts.
# Keep these commands commented unless you see a torchao-related error in Colab.
#import subprocess, sys
#subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'uninstall', '-y', 'torchao'], returncode=0)



After this optional cleanup, rerun the cell that failed. If you never see a `torchao` error, leave this cell commented and continue normally.


## 6 - Test the Fine-Tuned Model on One Candidate

This cell tests the trained model by giving it a sample candidate profile and scoring each allowed label. Instead of generating free text, it calculates the log-probability of `low_match`, `medium_match`, and `high_match`, then selects the label with the highest score.


In [ ]:
import torch
import torch.nn.functional as F

# These are the only labels the classifier is allowed to choose from.
labels = ["low_match", "medium_match", "high_match"]

# Example candidate profile used to test the fine-tuned model.
messages = [
    {"role": "system", "content":
     "You are a recruiting screening model. Return exactly one label: low_match, medium_match, or high_match."},
    {"role": "user", "content":
     "Candidate title: Data analyst with Python, SQL, Power BI, Tableau, and machine learning experience
"
     "Candidate location: United States
"
     "Label:"}
]

# Convert the chat messages to the same prompt format used by the tokenizer during training.
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
prompt_ids = tokenizer(prompt, return_tensors="pt").to(model.device)["input_ids"]

def score_label(lbl: str):
    # Score each possible label by summing the log-probabilities of its tokens.
    lbl_ids = tokenizer(" " + lbl, add_special_tokens=False, return_tensors="pt").to(model.device)["input_ids"]
    full = torch.cat([prompt_ids, lbl_ids], dim=1)

    with torch.no_grad():
        logits = model(full).logits  # [1, T, V]

    # log-prob of each label token conditioned on previous tokens
    logp = 0.0
    start = prompt_ids.shape[1]
    for i in range(lbl_ids.shape[1]):
        pos = start + i - 1
        token_id = lbl_ids[0, i]
        logp += F.log_softmax(logits[0, pos], dim=-1)[token_id].item()
    return logp

# Pick the label with the highest model score.
scores = {lbl: score_label(lbl) for lbl in labels}
pred = max(scores, key=scores.get)

print("Scores:", scores)
print("Final label:", pred)


Scores: {'low_match': -35.500755310058594, 'medium_match': -37.256317138671875, 'high_match': -33.503997802734375}
Final label: high_match




The notebook prints the score for each possible label and the final predicted label. This is a quick inference check that confirms the tokenizer, model, and trained adapter can be used together for candidate classification.
